In [ ]:
# --- Setup: make the `ecp` support package available -----------------
# Colab opens a single notebook and installs nothing, so fetch `ecp` from
# the public repo if it isn't importable yet. On Binder/local it is already
# installed, so this cell is a fast no-op there.
try:
    import ecp  # noqa: F401
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "git+https://github.com/ramador09/elementary-computational-physics-binder@main"],
        check=True,
    )


# 6.9 From Vectors to Wave Functions: The Position Representation and Continuous Spectra

<!-- This single H1 (one per notebook, "# <number> <Title>") is the page's
     title: it sets the sidebar entry, breadcrumb, browser tab, and search
     result. The branded banner below is generated by the shared `ecp`
     package, so the look of every notebook in the series lives in one place. -->

In [ ]:
from ecp.style import header, use_style

use_style()  # apply the series Matplotlib style
header(
    volume="Volume VI — Quantum Mechanics",
    number="6.9",
    title="From Vectors to Wave Functions: The Position Representation and Continuous Spectra",
    blurb="The same theory, infinitely many dimensions. A particle on a line has a "
    "state that is a function — the amplitude to be found at each point — and "
    "everything from the finite case carries over: sums become integrals, a "
    "Hermitian matrix becomes the operator −iℏ d/dx, and the change of basis to "
    "momentum is the Fourier transform. The price of the continuum is a new subtlety "
    "(eigenstates that cannot be normalized); the reward is that the uncertainty "
    "principle becomes Fourier reciprocity, and the Gaussian its sharpest state.",
    difficulty="advanced",
    estimate="160–200 min",
)

## Notebook overview

Every notebook so far has lived in a *finite* number of dimensions — two for a qubit, three or four
for the small examples. This notebook opens Movement II by taking the same formalism to **infinitely
many** dimensions, and the central message is one of reassurance: nothing here is a new postulate.
Wave mechanics is not a rival theory to the vectors and operators we have been using; it is *exactly*
that theory, written in a continuous basis. A particle on a line has a state $|\psi\rangle$ whose
"components" in the position basis form a **wave function** $\psi(x)=\langle x|\psi\rangle$ — the
amplitude to find the particle at $x$ — and every structure of Movements 0–I carries straight over.

The translation is worth seeing as a table, because each line is a finite object we already know
becoming its continuous cousin:

| finite-dimensional (Movements 0–I) | infinite-dimensional (here) |
| --- | --- |
| components $c_i=\langle e_i|\psi\rangle$ | wave function $\psi(x)=\langle x|\psi\rangle$ |
| inner product $\langle\phi|\psi\rangle=\sum_i\phi_i^{*}\psi_i$ | $\langle\phi|\psi\rangle=\int\phi^{*}(x)\psi(x)\,dx$ |
| normalization $\sum_i|c_i|^2=1$ | $\int|\psi(x)|^2\,dx=1$ (Born density) |
| resolution of the identity $\sum_i|e_i\rangle\langle e_i|=I$ | $\int|x\rangle\langle x|\,dx=I$ |
| a Hermitian matrix | a differential operator, $\hat p=-i\hbar\,d/dx$ |
| change of basis (a unitary) | the **Fourier transform** |

Two payoffs follow. The position and momentum operators obey the **canonical commutator** $[\hat x,
\hat p]=i\hbar$ — the continuous cousin of the Pauli non-commutation of [§6.6](pauli-uncertainty.ipynb) — and feeding it into the
Robertson relation of [§6.6](pauli-uncertainty.ipynb) *derives* the Heisenberg uncertainty principle $\Delta x\,\Delta p\ge
\hbar/2$, saturated by the **Gaussian** at every width (answering the forward pointer of [§6.6](pauli-uncertainty.ipynb)). And the
**Fourier transform** is revealed as nothing but the unitary change of basis between the position and
momentum representations, so a state narrow in $x$ is broad in $p$: the uncertainty principle *is*
Fourier reciprocity.

We are honest about the one genuinely new thing the continuum brings. The eigenstates of position and
momentum — sharp points and plane waves — are **not normalizable**: they are idealized limits, not
physical states, and this is exactly where the completeness we deferred in [§6.1](complex-vector-spaces.ipynb) finally bites. We name
the rigorous home (the rigged Hilbert space) without belaboring it, and keep the physics in the
normalizable **wave packets**.

As in every Volume VI notebook, each exercise opens with a **crystal-clear statement** and enumerated parts, each naming the exact operation — a discretized **grid**, `numpy.fft.fft`/`ifft`/`fftfreq` for
the Fourier transform and the spectral-derivative momentum operator (with the normalization stated),
and `numpy.trapezoid` for the expectation integrals.

> **Conventions.** We set $\hbar=1$. The grid is $N$ points on $[-L/2,L/2)$ with spacing
> $dx=L/N$; the conjugate (angular) wavenumber grid is $k=2\pi\,$`numpy.fft.fftfreq(N, dx)` and the
> momentum is $p=\hbar k$. The momentum operator is the **spectral derivative**, $\hat p\psi=$`ifft(ℏk·
> fft(ψ))`. All numerical checks are made in the **bulk**, away from the periodic grid boundaries
> where the spectral derivative wraps around. The Schrödinger equation solved on this grid is [§6.10](schrodinger-on-a-computer.ipynb);
> coherent states are [§6.12](harmonic-oscillator.ipynb); wave-packet dynamics is [§6.13](scattering-tunneling.ipynb). See Sakurai & Napolitano (§1.6–1.7);
> Nolting; and Notebooks [§6.1](complex-vector-spaces.ipynb) (states, completeness), [§6.2](operators-spectral-theorem.ipynb) (operators), [§6.3](dirac-notation-spectral-decomposition.ipynb) (change of basis), [§6.5](postulates.ipynb) (the
> Born rule), [§6.6](pauli-uncertainty.ipynb) (the uncertainty relation), and Volume II (canonical quantization).

## Theory in brief

### The wave function as continuous components

A particle on a line has a state whose components in the continuous **position basis** $\{|x\rangle\}$
form the **wave function**,

```{math}
:label: eq-wavefunction
\psi(x)=\langle x|\psi\rangle,\qquad \langle\phi|\psi\rangle=\int\phi^{*}(x)\psi(x)\,dx,\qquad \int|\psi(x)|^2\,dx=1,\qquad \int|x\rangle\langle x|\,dx=I .
```

Born's rule becomes a **probability density**: $|\psi(x)|^2\,dx$ is the probability of finding the
particle in $[x,x+dx]$ — the continuous version of the $|c_i|^2$ of [§6.1](complex-vector-spaces.ipynb) and [§6.5](postulates.ipynb).

### The position and momentum operators

In the position representation, position acts by **multiplication** and momentum by
**differentiation** — the derivative being what the generator of translations must look like on wave
functions, an identification Sakurai & Napolitano (§1.6) make precise:

```{math}
:label: eq-xp-operators
(\hat x\psi)(x)=x\,\psi(x),\qquad (\hat p\psi)(x)=-i\hbar\,\frac{d\psi}{dx} ,
```

both Hermitian (with decay at infinity). On a computer $\hat x$ multiplies by the grid, and $\hat p$
is the **spectral derivative** — multiply by $\hbar k$ in Fourier space — the infinite-dimensional
Hermitian operators of [§6.2](operators-spectral-theorem.ipynb).

### The canonical commutator

Apply $\hat x\hat p-\hat p\hat x$ to any wave function and the product rule does the rest:
$-i\hbar\left[x\psi'-(x\psi)'\right]=i\hbar\,\psi$, one line of calculus from the operators above
(§1.6 of Sakurai & Napolitano derives it basis-independently, from translations),

```{math}
:label: eq-canonical
[\hat x,\hat p]=i\hbar ,
```

the foundational relation of quantum mechanics: the continuous analogue of $[\sigma_x,\sigma_y]=2i
\sigma_z$, and what canonical quantization promotes the classical Poisson bracket $\{x,p\}=1$ into
(Volume II; $\hbar\to0$ recovers classical mechanics). Everything non-classical flows from it.

### The position–momentum uncertainty relation

Feeding $[\hat x,\hat p]=i\hbar$ into the Robertson relation of [§6.6](pauli-uncertainty.ipynb) gives the **Heisenberg uncertainty
principle**, now *derived* rather than postulated,

```{math}
:label: eq-xp-uncertainty
\Delta x\,\Delta p\ \ge\ \frac{\hbar}{2},\qquad \text{saturated by the Gaussian } \psi(x)\propto e^{-x^2/4\sigma^2}\ \text{at every width} .
```

The Gaussian is the minimum-uncertainty state — the wave-mechanical cousin of the spin $|{+}z\rangle$
of [§6.6](pauli-uncertainty.ipynb), and the seed of coherent states ([§6.12](harmonic-oscillator.ipynb)).

### The Fourier transform as the momentum representation

The momentum-space wave function is the **Fourier transform** of $\psi(x)$,

```{math}
:label: eq-fourier
\varphi(p)=\langle p|\psi\rangle=\frac{1}{\sqrt{2\pi\hbar}}\int\psi(x)\,e^{-ipx/\hbar}\,dx ,
```

and the momentum basis $\{|p\rangle\}$ is just another orthonormal basis. The Fourier transform is the
**unitary change of basis** between position and momentum ([§6.3](dirac-notation-spectral-decomposition.ipynb)); Parseval's theorem is its
norm-preservation. Conjugate widths ($\Delta p\approx\hbar/2\Delta x$) make the uncertainty principle
literally **Fourier reciprocity**.

### Continuous spectra and non-normalizable eigenstates

The genuinely new feature. The eigenstates of $\hat x$ (points $|x_0\rangle$) and of $\hat p$ (plane
waves $e^{ipx/\hbar}$, the de Broglie waves with $\lambda=2\pi\hbar/p$) are **not** square-integrable,

```{math}
:label: eq-continuous
\hat p\,e^{ip_0x/\hbar}=p_0\,e^{ip_0x/\hbar},\qquad \int|e^{ip_0x/\hbar}|^2dx=\infty,\qquad \langle x|x'\rangle=\delta(x-x'),\ \langle p|p'\rangle=\delta(p-p') .
```

They are idealized limits, $\delta$-normalized — and here the completeness deferred from [§6.1](complex-vector-spaces.ipynb) finally
bites. The rigorous home is the **rigged Hilbert space** (named, not developed); physical states are
normalizable **wave packets**.

## Setup

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from ecp import draw, validate

ACCENT, INK, SOFT = draw.ACCENT, draw.INK, draw.SOFT
RED = "#c1121f"

HBAR = 1.0  # ℏ = 1 throughout

# A discretized grid on [−L/2, L/2): N points, spacing dx. The conjugate wavenumber grid is
# k = 2π·numpy.fft.fftfreq(N, dx); the momentum grid is p = ℏk. The momentum operator is the spectral
# derivative p̂ψ = ifft(ℏk·fft(ψ)) (the FFT normalization: numpy's fft/ifft are an inverse pair, so the
# round trip carries no extra factor). All checks are made in the BULK, away from the periodic edges.

N_GRID = 2048
L_BOX = 40.0
DX = L_BOX / N_GRID
X_GRID = -L_BOX / 2 + DX * np.arange(N_GRID)


def trapz(values, x):
    """Integrate ``values`` over the grid ``x`` with ``numpy.trapezoid`` (the trapezoidal rule)."""
    return np.trapezoid(values, x)


def normalize(psi, x):
    """Return ``psi`` scaled so that $\\int|\\psi(x)|^2\\,dx=1$ {eq}`eq-wavefunction`.

    Divides by $\\sqrt{\\int|\\psi|^2dx}$ (the continuous version of dividing by the norm in §6.1); after
    this $|\\psi(x)|^2$ is a genuine probability density.
    """
    return psi / np.sqrt(trapz(np.abs(psi) ** 2, x))


def x_operator(psi, x):
    """The position operator $\\hat x\\psi=x\\,\\psi(x)$ {eq}`eq-xp-operators` — multiplication by the grid."""
    return x * psi


def p_operator(psi, x):
    """The momentum operator $\\hat p\\psi=-i\\hbar\\,d\\psi/dx$ by the **spectral derivative** {eq}`eq-xp-operators`.

    Differentiation is exact (to machine precision) in Fourier space: $\\hat p$ multiplies by $\\hbar k$
    there, so $\\hat p\\psi=$``numpy.fft.ifft(ℏk·numpy.fft.fft(psi))`` with $k=2\\pi\\,$``numpy.fft.fftfreq(
    N, dx)``. The result is exact in the bulk and wraps around at the periodic grid edges.

    Parameters
    ----------
    psi : numpy.ndarray
        The wave function sampled on ``x``.
    x : numpy.ndarray
        A uniform grid.

    Returns
    -------
    numpy.ndarray
        $\\hat p\\psi$ on the grid.
    """
    n = len(x)
    dx = x[1] - x[0]
    k = 2 * np.pi * np.fft.fftfreq(n, d=dx)
    return np.fft.ifft(HBAR * k * np.fft.fft(psi))


def to_momentum(psi, x):
    """The momentum-space wave function $\\varphi(p)=\\langle p|\\psi\\rangle$ {eq}`eq-fourier`.

    The normalized Fourier transform $\\varphi(p)=(2\\pi\\hbar)^{-1/2}\\int\\psi(x)e^{-ipx/\\hbar}dx$,
    evaluated with `numpy.fft.fft`. The grid offset $x_0$ contributes a phase $e^{-ipx_0/\\hbar}$, and
    the factor $dx/\\sqrt{2\\pi\\hbar}$ makes the transform **unitary** (Parseval: the norm is preserved).
    Returns $\\varphi$ and the (ascending) momentum grid $p=\\hbar k$.

    Parameters
    ----------
    psi : numpy.ndarray
        The position-space wave function.
    x : numpy.ndarray
        The position grid.

    Returns
    -------
    phi : numpy.ndarray
        The momentum-space wave function on the ascending ``p`` grid.
    p : numpy.ndarray
        The momentum grid (ascending).
    """
    n = len(x)
    dx = x[1] - x[0]
    x0 = x[0]
    k = 2 * np.pi * np.fft.fftfreq(n, d=dx)
    p = HBAR * k
    phi = dx / np.sqrt(2 * np.pi * HBAR) * np.exp(-1j * p * x0 / HBAR) * np.fft.fft(psi)
    order = np.argsort(p)
    return phi[order], p[order]


def expectation(op, psi, x):
    """The expectation value $\\langle\\psi|\\hat O|\\psi\\rangle=\\int\\psi^{*}(\\hat O\\psi)\\,dx$ (§6.5).

    Applies the operator ``op(psi, x)`` and integrates against $\\psi^{*}$ with `numpy.trapezoid`; the
    result is real for a Hermitian operator.
    """
    return trapz(np.conj(psi) * op(psi, x), x).real


def uncertainty(op, psi, x):
    """The uncertainty $\\Delta O=\\sqrt{\\langle\\hat O^2\\rangle-\\langle\\hat O\\rangle^2}$ (§6.5, §6.6).

    Applies ``op`` twice for $\\langle\\hat O^2\\rangle$. For $\\hat x$ and the spectral $\\hat p$ this
    gives $\\Delta x$ and $\\Delta p$, whose product the uncertainty relation bounds.
    """
    mean = expectation(op, psi, x)
    mean_sq = trapz(np.conj(psi) * op(op(psi, x), x), x).real
    return np.sqrt(max(mean_sq - mean**2, 0.0))


def gaussian_packet(x, x0=0.0, p0=0.0, sigma=1.0):
    """A normalized Gaussian wave packet $\\propto e^{-(x-x_0)^2/4\\sigma^2}e^{ip_0x/\\hbar}$ {eq}`eq-xp-uncertainty`.

    Centered at $x_0$ with mean momentum $p_0$ and position width $\\sigma$ ($\\Delta x=\\sigma$). It is
    the minimum-uncertainty state, $\\Delta x\\,\\Delta p=\\hbar/2$.
    """
    psi = np.exp(-((x - x0) ** 2) / (4 * sigma**2)) * np.exp(1j * p0 * x / HBAR)
    return normalize(psi, x)

## Exercise 1 — The wave function and its probability density

A particle on a line is described by a wave function $\psi(x)$ on a grid. Normalize it so that
$\int|\psi(x)|^2\,dx=1$, interpret $|\psi(x)|^2$ as a probability density, and compute the
probability of finding the particle in a region $[a,b]$ {eq}`eq-wavefunction`.

1. Set up the grid (`X_GRID`) and build a wave packet (a `gaussian_packet`).
2. Normalize it so $\int|\psi|^2dx=1$ with `numpy.trapezoid` (the `normalize` helper).
3. Read $|\psi(x)|^2$ as the Born **probability density** — the continuous $|c_i|^2$ of [§6.1](complex-vector-spaces.ipynb)/[§6.5](postulates.ipynb).
4. Compute $P(a<x<b)=\int_a^b|\psi|^2dx$ by integrating the density over the sub-grid with
   `numpy.trapezoid`. Frame: $\psi(x)=\langle x|\psi\rangle$ are the components of $|\psi\rangle$
   in the position basis.

In [ ]:
# (solution hidden on the public site)


### Validation 1

In [ ]:
validate.close(
    total,
    1.0,
    "the wave function is normalized, ∫|ψ(x)|²dx=1; |ψ(x)|² is a probability density (the continuous Born rule)",
    rtol=1e-6,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 2 — The position and momentum operators

Implement the position operator $\hat x$ (multiplication by the grid) and the momentum operator
$\hat p=-i\hbar\,d/dx$ (the spectral derivative), and compute their expectation values $\langle
x\rangle$ and $\langle p\rangle$ for a wave packet, confirming both are real
{eq}`eq-xp-operators`.

1. The position operator is the `x_operator` helper, $\hat x\psi=x\cdot\psi$.
2. The momentum operator is the `p_operator` helper, the spectral derivative $\hat
   p\psi=$`numpy.fft.ifft(ℏk· numpy.fft.fft(psi))` with $k=2\pi\,$`numpy.fft.fftfreq(N, dx)`.
3. Compute $\langle x\rangle$ and $\langle p\rangle$ with the `expectation` helper
   ($\int\psi^{*}\hat O\psi\,dx$ via `numpy.trapezoid`).
4. Confirm both are real (the imaginary parts vanish — Hermitian operators) and match the packet's
   centre $x_0$ and mean momentum $p_0$. Frame: the infinite-dimensional Hermitian operators of
   [§6.2](operators-spectral-theorem.ipynb).

In [ ]:
# (solution hidden on the public site)


### Validation 2

In [ ]:
validate.check(
    np.isclose(x_mean, -2.0, atol=1e-3)
    and np.isclose(p_mean, 1.5, atol=1e-3)
    and abs(x_imag) < 1e-9
    and abs(p_imag) < 1e-9,
    "position multiplies and momentum differentiates (p̂=−iℏd/dx via the spectral derivative); both ⟨x⟩ and ⟨p⟩ are real",
)

## Exercise 3 — The canonical commutator $[\hat x,\hat p]=i\hbar$

Verify the foundational relation of quantum mechanics: $[\hat x,\hat p]\psi=i\hbar\psi$ for a
smooth test function, in the bulk of the grid {eq}`eq-canonical`.

1. Take a smooth wave packet and compute $\hat x\hat p\psi$ (apply `p_operator` then multiply by
   $x$) and $\hat p\hat x\psi$ (multiply by $x$ then apply `p_operator`).
2. Form the commutator $[\hat x,\hat p]\psi=\hat x\hat p\psi-\hat p\hat x\psi$.
3. Confirm it equals $i\hbar\psi$ in the **bulk** (the middle half of the grid, away from the
   periodic edges where the spectral derivative wraps) by checking the ratio $[\hat x,\hat
   p]\psi/(i\hbar\psi)\approx1$.
4. Note this is the continuous analogue of the Pauli non-commutation of [§6.6](pauli-uncertainty.ipynb) and the quantization
   of $\{x,p\}=1$ (Volume II). Frame: everything non-classical flows from this.

In [ ]:
# (solution hidden on the public site)


### Validation 3

In [ ]:
validate.close(
    ratio.mean().real,
    1.0,
    "the canonical commutator [x̂,p̂]=iℏ holds (verified in the bulk of the grid)",
    rtol=1e-3,
)

## Exercise 4 — The position–momentum uncertainty relation

Compute $\Delta x\cdot\Delta p$ for a wave packet and confirm the Heisenberg uncertainty relation
$\Delta x\,\Delta p\ge\hbar/2$ — now *derived* from the canonical commutator through the Robertson
relation of [§6.6](pauli-uncertainty.ipynb), not postulated {eq}`eq-xp-uncertainty`.

1. Compute $\Delta x=\sqrt{\langle\hat x^2\rangle-\langle\hat x\rangle^2}$ with the `uncertainty`
   helper applied to `x_operator`.
2. Compute $\Delta p$ likewise with `p_operator` (the spectral derivative).
3. Form the product $\Delta x\cdot\Delta p$ and confirm it is $\ge\hbar/2$.
4. Connect: Robertson ([§6.6](pauli-uncertainty.ipynb)) with $[\hat x,\hat p]=i\hbar$ gives $\Delta x\,\Delta
   p\ge\tfrac12|\langle[ \hat x,\hat p]\rangle|=\hbar/2$. Frame: Heisenberg's principle is a
   theorem about the commutator.

In [ ]:
# (solution hidden on the public site)


### Validation 4

In [ ]:
validate.check(
    dx_c * dp_c >= HBAR / 2 - 1e-9
    and all(
        uncertainty(x_operator, gaussian_packet(X_GRID, sigma=s), X_GRID)
        * uncertainty(p_operator, gaussian_packet(X_GRID, sigma=s), X_GRID)
        >= HBAR / 2 - 1e-6
        for s in (0.8, 1.4, 2.5)
    ),
    "the position–momentum uncertainty relation Δx·Δp ≥ ℏ/2 follows from [x̂,p̂]=iℏ (Robertson, §6.6)",
)

## Exercise 5 — The Gaussian: the minimum-uncertainty state

Show that the Gaussian saturates the uncertainty relation, $\Delta x\,\Delta p=\hbar/2$, for *any*
width — the position and momentum spreads trade off, but their product is fixed at the minimum
{eq}`eq-xp-uncertainty`.

1. Build Gaussians $\psi\propto e^{-x^2/4\sigma^2}$ of several widths $\sigma$ with
   `gaussian_packet`.
2. For each, compute $\Delta x$ and $\Delta p$ with the `uncertainty` helper.
3. Confirm $\Delta x=\sigma$, $\Delta p=\hbar/2\sigma$, and the product $\Delta x\cdot\Delta
   p=\hbar/2$ exactly.
4. Note the trade-off: narrow in $x$ means broad in $p$, but the product never beats $\hbar/2$.
   Frame: the wave-mechanical analogue of the spin $|{+}z\rangle$ of [§6.6](pauli-uncertainty.ipynb), and the seed of coherent
   states ([§6.12](harmonic-oscillator.ipynb)).

In [ ]:
# (solution hidden on the public site)


### Validation 5

In [ ]:
validate.close(
    products,
    np.full_like(products, HBAR / 2),
    "the Gaussian is the minimum-uncertainty state: Δx·Δp = ℏ/2 at every width",
    atol=2e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 6 — The Fourier transform as the momentum representation

Compute the momentum-space wave function $\varphi(p)=\langle p|\psi\rangle$ and verify that the
Fourier transform is the **unitary** change of basis between position and momentum: it preserves
the norm (Parseval), and $\langle p\rangle$ computed in either representation agrees
{eq}`eq-fourier`.

1. Compute $\varphi(p)$ with the `to_momentum` helper — the normalized Fourier transform
   $(2\pi\hbar)^{-1/2}\int\psi e^{-ipx/\hbar}dx$ via `numpy.fft.fft`, on the momentum grid
   $p=\hbar k$.
2. Verify **Parseval**: $\int|\varphi(p)|^2dp=\int|\psi(x)|^2dx=1$ (`numpy.trapezoid`).
3. Verify $\langle p\rangle$ agrees: from the position representation ($\int\psi^{*}\hat
   p\psi\,dx$, the `expectation` of `p_operator`) and from the momentum representation ($\int
   p\,|\varphi(p)|^2dp$).
4. Show conjugate widths: a narrow packet in $x$ is broad in $p$. Frame: the Fourier transform is
   the unitary change of basis ([§6.3](dirac-notation-spectral-decomposition.ipynb)); the uncertainty principle is Fourier reciprocity.

In [ ]:
# (solution hidden on the public site)


### Validation 6

In [ ]:
validate.close(
    [norm_p, p_from_x],
    [1.0, p_from_p],
    "the Fourier transform is the unitary position↔momentum change of basis: Parseval (norm preserved) and ⟨p⟩ agree across representations",
    rtol=1e-2,
)

In [ ]:
# (solution hidden on the public site)


## Exercise 7 — Momentum eigenstates, de Broglie, and the continuous spectrum *(student)*

Show that a plane wave $\psi(x)=e^{ip_0x/\hbar}$ is an eigenstate of the momentum operator, find
its de Broglie wavelength, and explain honestly why it is *not* a physical state — the genuinely
new feature of the continuum {eq}`eq-continuous`.

1. Build the plane wave $\psi(x)=e^{ip_0x/\hbar}$ on the grid.
2. Apply the `p_operator` and confirm in the bulk that $\hat p\psi=p_0\psi$ — an eigenvalue $p_0$.
3. Compute the de Broglie wavelength $\lambda=2\pi\hbar/p_0$.
4. Show $\int|\psi|^2dx$ grows without bound as the box grows ($|\psi|^2=1$ everywhere), so the
   plane wave is **non-normalizable** — a continuous-spectrum eigenstate, $\delta$-normalized,
   living in the rigged Hilbert space (named, not developed).
5. Note that physical states are normalizable wave packets, superpositions of plane waves. Frame:
   where the completeness deferral of [§6.1](complex-vector-spaces.ipynb) finally bites.

In [ ]:
# (solution hidden on the public site)


### Validation 7

In [ ]:
validate.close(
    eig_ratio,
    p0,
    "a plane wave e^{ip₀x/ℏ} is a momentum eigenstate (de Broglie); it is non-normalizable — the continuous spectrum, δ-normalized (the rigged Hilbert space)",
    rtol=1e-2,
)

## Exercise 8 — Wave mechanics is the same mechanics

Nothing in this notebook was a new postulate. The **wave function** is the state vector's components
in the position basis, $\psi(x)=\langle x|\psi\rangle$; the **inner product** is an integral; the
**momentum operator** is a derivative, $\hat p=-i\hbar\,d/dx$; and the **change to momentum** is a
Fourier transform. Every line of the bridge table from the overview is a finite object we already had,
grown to a continuum. The one genuinely new thing — the **continuous spectrum** — cost us
normalizability for the idealized position and momentum eigenstates and gave us, in return, the
uncertainty principle as **Fourier reciprocity** and the **Gaussian** as its sharpest state.

**This exercise (synthesis).** No new computation: the correspondence is the result. Schrödinger's
wave mechanics and Heisenberg's matrices felt like rival theories for a year in 1926; they are the
*same* theory in two bases, and the Fourier transform is the unitary that connects them — a fact we
just verified on a grid. With the position representation in hand, the next notebook ([§6.10](schrodinger-on-a-computer.ipynb)) writes the
Schrödinger equation as a differential equation, discretizes it into a matrix, and solves it the way
we solved every operator problem in Movement 0 — by **diagonalizing**. The arena has changed from two
amplitudes to a continuum of them; the method has not changed at all.

## Notebook summary

Wave mechanics as the same formalism in infinitely many dimensions — the opening of Movement II.

- **The wave function** {eq}`eq-wavefunction`: $\psi(x)=\langle x|\psi\rangle$, the components in the
  position basis; $\int|\psi|^2dx=1$ with $|\psi(x)|^2$ a Born density; sums become integrals and the
  resolution of the identity becomes $\int|x\rangle\langle x|dx=I$.
- **The operators** {eq}`eq-xp-operators`: $\hat x$ multiplies by the grid, $\hat p=-i\hbar\,d/dx$ is
  the spectral derivative (`numpy.fft`) — the infinite-dimensional Hermitian operators of [§6.2](operators-spectral-theorem.ipynb).
- **The canonical commutator** {eq}`eq-canonical`: $[\hat x,\hat p]=i\hbar$ (verified in the bulk) —
  the continuous cousin of the Pauli non-commutation, the quantization of $\{x,p\}=1$.
- **Uncertainty** {eq}`eq-xp-uncertainty`: $\Delta x\,\Delta p\ge\hbar/2$ *derived* from the
  commutator via Robertson ([§6.6](pauli-uncertainty.ipynb)), saturated by the **Gaussian** at every width.
- **The Fourier transform** {eq}`eq-fourier`: the unitary position↔momentum change of basis
  (Parseval; $\langle p\rangle$ agrees both ways) — the uncertainty principle as Fourier reciprocity.
- **The continuous spectrum** {eq}`eq-continuous`: plane waves are non-normalizable, $\delta$-normalized
  eigenstates (the rigged Hilbert space); physical states are wave packets — where the completeness
  deferral of [§6.1](complex-vector-spaces.ipynb) finally bites.

Wave mechanics is the same mechanics. The next notebook diagonalizes the Schrödinger operator on a
grid; the method is the one from Movement 0, only the arena is new.

## Outlook

- **The Schrödinger equation on a grid ([§6.10](schrodinger-on-a-computer.ipynb))**: the differential equation discretized to a matrix
  eigenproblem and solved by diagonalization — bound states and spectra.
- **One-dimensional systems ([§6.11](bound-states-1d.ipynb)), the harmonic oscillator and coherent states ([§6.12](harmonic-oscillator.ipynb)), wave-packet
  dynamics ([§6.13](scattering-tunneling.ipynb))**: split-step Fourier and Crank–Nicolson propagation.
- **The rigged Hilbert space and the spectral theorem for continuous spectra** (a horizon, named).
- **Cross-reference** [§6.1](complex-vector-spaces.ipynb) (states / completeness), [§6.2](operators-spectral-theorem.ipynb) (operators), [§6.3](dirac-notation-spectral-decomposition.ipynb) (change of basis), [§6.5](postulates.ipynb) (the
  Born rule), [§6.6](pauli-uncertainty.ipynb) (uncertainty), Volume II (canonical quantization), and forward to [§6.10](schrodinger-on-a-computer.ipynb)–[§6.13](scattering-tunneling.ipynb).

In [ ]:
from ecp.style import footer

footer()